In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Clean uninstall everything
!pip uninstall -y trl transformers peft accelerate bitsandbytes numpy

# Downgrade NumPy to avoid PyTorch incompatibility
!pip install numpy==1.26.4

# Install PyTorch with CUDA 11.8
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

# Install ONLY compatible versions
!pip install --no-cache-dir \
  trl==0.8.6 \
  transformers==4.41.2 \
  peft==0.10.0 \
  accelerate==0.26.1 \
  bitsandbytes==0.42.0 \
  datasets==2.16.1 \
  evaluate pillow


Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: accelerate 0.26.1
Uninstalling accelerate-0.26.1:
  Successfully uninstalled accelerate-0.26.1
Found existing installation: bitsandbytes 0.42.0
Uninstalling bitsandbytes-0.42.0:
  Successfully uninstalled bitsandbytes-0.42.0
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [2]:
import transformers, peft, trl
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)


/usr/local/lib/python3.11/dist-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


/usr/local/lib/python3.11/dist-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32
Transformers: 4.41.2
PEFT: 0.10.0
TRL: 0.8.6


In [4]:
import os
import json

# Step 1: Define your formatting function (output ONLY prompt, chosen, rejected)
def formatdata(data):
    prompt = (
        f"Given this dialogue with a user contradiction, the contradiction happens between "
        f"{data['Contradiction']['curr_pref']} and {data['Contradiction']['prev_pref']} "
        f"with certainty level {data['Contradiction']['certainty']}. "
        f"Generate the response."
    )
    return {
        'prompt': prompt,
        'chosen': data['Response']['preferred_response'],
        'rejected': data['Response']['dispreferred_response']
    }

# Step 2: Safe wrapper for formatting
def safe_format(entry, file_path):
    try:
        return formatdata(entry)
    except Exception as e:
        print(f"⚠️ Error formatting entry in {file_path}: {e}")
        return None

# Step 3: Root directory (update this to your Google Drive folder path)
root_dir = '/content/drive/MyDrive/output'

# Step 4: Storage
formatted_data = []
file_count = 0

# Step 5: Recursive walk through subfolders
for subdir, _, files in os.walk(root_dir):
    print(f"🔍 Searching in: {subdir}")
    for filename in files:
        if filename.endswith('.json'):
            file_count += 1
            file_path = os.path.join(subdir, filename)
            print(f"📂 Processing: {file_path}")
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                    # Since each file is a single dictionary
                    if isinstance(data, dict):
                        formatted = safe_format(data, file_path)
                        if formatted:
                            formatted_data.append(formatted)
            except Exception as e:
                print(f"❌ Failed to load JSON from {file_path}: {e}")
        else:
            print(f"⏩ Skipping non-JSON file: {filename}")

# Step 6: Save all formatted outputs to a JSONL file
output_path = 'all_formatted_data.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for item in formatted_data:
        f.write(json.dumps(item) + '\n')

print(f"\n📄 Total JSON files processed: {file_count}")
print(f"✅ Saved {len(formatted_data)} formatted items to {output_path}")


🔍 Searching in: /content/drive/MyDrive/output
🔍 Searching in: /content/drive/MyDrive/output/foodRecommendation
📂 Processing: /content/drive/MyDrive/output/foodRecommendation/conversation_foodRecommendation_persona0_sample0.json
🔍 Searching in: /content/drive/MyDrive/output/travelPlanning
📂 Processing: /content/drive/MyDrive/output/travelPlanning/conversation_travelPlanning_persona0_sample0.json
🔍 Searching in: /content/drive/MyDrive/output/studyConsultation
📂 Processing: /content/drive/MyDrive/output/studyConsultation/conversation_studyConsultation_persona0_sample0.json
🔍 Searching in: /content/drive/MyDrive/output/datingConsultation
📂 Processing: /content/drive/MyDrive/output/datingConsultation/conversation_datingConsultation_persona0_sample0.json
🔍 Searching in: /content/drive/MyDrive/output/financialConsultation
📂 Processing: /content/drive/MyDrive/output/financialConsultation/conversation_financialConsultation_persona0_sample0.json
🔍 Searching in: /content/drive/MyDrive/output/home

In [36]:
from datasets import Dataset
dataset = Dataset.from_list(formatted_data)
dataset = dataset.train_test_split(test_size=0.1)

In [35]:
for i, example in enumerate(dataset["train"]):
    print(f"\n--- Example {i} ---")
    print("🟡 Prompt:\n", example["prompt"])
    print("✅ Chosen:\n", example["chosen"])
    print("❌ Rejected:\n", example["rejected"])



--- Example 0 ---
🟡 Prompt:
 Given this dialogue with a user contradiction, the contradiction happens between open to vegan Indian food and not a fan of spicy food with certainty level high. Generate the response.
✅ Chosen:
 It's great to hear you’re interested in vegan Indian cuisine! I can suggest some excellent local restaurants that specialize in that.
❌ Rejected:
 I see you're still unsure about Indian food. Maybe stick to what you know, like poke and taro.

--- Example 1 ---
🟡 Prompt:
 Given this dialogue with a user contradiction, the contradiction happens between integrating theory with practice and keeping theory and practice separate with certainty level high. Generate the response.
✅ Chosen:
 I'm glad you're open to integrating theory with practice! Tracking your progress can be done with self-assessments and reflecting on how well you apply what you've learned.
❌ Rejected:
 I think you should stick to your initial preference of keeping theory and practice separate, as it w

In [37]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

model_id = "Qwen/Qwen1.5-1.8B"

# 1. Load base model (not meta)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",  # or manually map to cuda
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# 2. Define your LoRA config
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # adjust for Qwen's architecture
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 3. Apply LoRA
model = get_peft_model(model, peft_config)


In [38]:
prompt_length = 512
max_seq_length = 1512
import torch
!nvidia-smi
!nvcc --version
print(torch.__version__)
#!pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html


/bin/bash: line 1: nvidia-smi: command not found
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
2.1.2+cu118


In [40]:
#from trl.trainer import DPOConfig
import transformers
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="doplhin-dpo",               # directory to save and repository id
    num_train_epochs=1,                     # number of training epochs
    per_device_train_batch_size=12,         # batch size per device during training
    per_device_eval_batch_size=4,           # batch size for evaluation
    gradient_accumulation_steps=1,          # number of steps before performing a backward/update pass
    gradient_checkpointing=True,            # use gradient checkpointing to save memory
    optim="adamw_torch",              # use fused adamw optimizer
    learning_rate=5e-5,                     # 10x higher LR than QLoRA paper
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.1,                       # warmup ratio based on QLoRA paper
    lr_scheduler_type="cosine",             # use cosine learning rate scheduler
    logging_steps=25,                       # log every 25 steps
    save_steps=500,                         # when to save checkpoint
    save_total_limit=2,                     # limit the total amount of checkpoints
    evaluation_strategy="steps",            # evaluate every 1000 steps
    eval_steps=700,                         # when to evaluate
    bf16=True,                              # use bfloat16 precision
    tf32=False,                              # use tf32 precision
    push_to_hub=False,                      # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
)

dpo_args = {
    "beta": 0.1,                            # The beta factor in DPO loss. Higher beta means less divergence
    "loss_type": "sigmoid"                  # The loss type for DPO.
}



/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [41]:
from trl import DPOTrainer

trainer = DPOTrainer(
    model,
    ref_model=None, # set to none since we use peft
    peft_config=peft_config,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
    max_length=max_seq_length,
    max_prompt_length=prompt_length,
    beta=dpo_args["beta"],
    loss_type=dpo_args["loss_type"],
)

/usr/local/lib/python3.11/dist-packages/trl/trainer/dpo_trainer.py:332: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [42]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model at the end of training
trainer.save_model()

TypeError: 'NoneType' object cannot be interpreted as an integer

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model at the end of training
trainer.save_model()